In [38]:
using LowLevelFEM, LinearAlgebra, SparseArrays
using IterativeSolvers, Preconditioners

In [39]:
spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "spectral.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

┌ Warning: Replacing docs for `LowLevelFEM._sem_legendre_pair :: Tuple{Integer, Real}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_gll :: Tuple{Integer}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_barycentric_weights :: Tuple{AbstractVector}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_lagrange_matrix :: Tuple{AbstractVector, AbstractVector}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253


┌ Warning: Replacing docs for `LowLevelFEM._sem_finalize_column_pattern! :: Tuple{Vector{Vector{Int64}}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_csc_from_column_pattern :: Tuple{Integer, Integer, Vector{Vector{Int64}}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_set_unique! :: Tuple{SparseMatrixCSC{Float64, Int64}, Integer, Integer, Real}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_nodal_prolongation :: Tuple{Problem, Any, Any}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_spectral_owners :: Tuple{Any, Any}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_nodal_restriction :: Tuple{Problem, Any, Any}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevel

In [40]:
openGeometry("plate-with-hole.geo")
#openPreProcessor()

In [41]:
mat = Material("body")
P1 = Field([mat], type=:VectorField, dim=2, fieldName=:u)
P2 = Field([mat], type=:VectorField, dim=2, fieldName=:u, basis=:spectral);

In [42]:
K1 = ∫(SymGrad(P1) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(P1))
f1 = ∫(P1 ⋅ [1,0], Γ="right")

K2 = ∫(SymGrad(P2) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(P2))
f2 = ∫(P2 ⋅ [1,0], Γ="right");

In [43]:
bc = BoundaryCondition("left", ux=0, uy=0);

In [56]:
@time u1 = solveField(Symmetric(K1), f1, support=[bc])
@time u2 = solveField(Symmetric(K2), f2, support=[bc])

err = norm(DoFs(u1) - DoFs(u2)) / norm(DoFs(u1))

  0.474480 seconds (32.03 k allocations: 300.226 MiB, 40.31% gc time)
  0.380172 seconds (31.60 k allocations: 859.269 MiB, 24.65% gc time)


1.913045414181366e-13

In [57]:
@time u1 = solveField(Symmetric(K1), f1, support=[bc], solver=:cg)
@time u2 = solveField(Symmetric(K2), f2, support=[bc], solver=:cg)

err = norm(DoFs(u1) - DoFs(u2)) / norm(DoFs(u1))

  1.357424 seconds (35.42 k allocations: 212.245 MiB, 0.56% gc time)
  1.532623 seconds (34.87 k allocations: 771.573 MiB, 6.07% gc time)


1.519082432579846e-9

In [46]:
p1 = LowLevelFEM.prepare_singlefield_system(
    Symmetric(K1), f1, [bc]
)

p2 = LowLevelFEM.prepare_singlefield_system(
    Symmetric(K2), f2, [bc]
)

@time x1 = LowLevelFEM.solve_linear_system(
    p1.A, p1.B;
    solver=:cg
)

@time x2 = LowLevelFEM.solve_linear_system(
    p2.A, p2.B;
    solver=:cg
);

  1.291607 seconds (3.47 k allocations: 77.820 MiB, 6.02% gc time)
  1.255692 seconds (3.35 k allocations: 77.798 MiB, 0.24% gc time)


In [47]:
tol = sqrt(eps())

@time x1, hist1 = cg(
    p1.A,
    @view(p1.B[:, 1]);
    reltol=tol,
    maxiter=size(p1.A, 1),
    log=true
)

@time x2, hist2 = cg(
    p2.A,
    @view(p2.B[:, 1]);
    reltol=tol,
    maxiter=size(p2.A, 1),
    log=true
)

@show hist1.iters
@show hist2.iters

 17.345711 seconds (22.12 k allocations: 1.651 MiB)
 11.112986 seconds (12.98 k allocations: 1.455 MiB)
hist1.iters = 4511
hist2.iters = 2689


2689

In [48]:
A1 = p1.A isa Symmetric ? parent(p1.A) : p1.A
A2 = p2.A isa Symmetric ? parent(p2.A) : p2.A

@show size(A1)
@show size(A2)

@show nnz(A1)
@show nnz(A2)
@show nnz(A2) / nnz(A1)

size(A1) = (30774, 30774)
size(A2) = (30774, 30774)
nnz(A1) = 2168156
nnz(A2) = 2168156
nnz(A2) / nnz(A1) = 1.0


1.0

In [49]:
Ts, Rs = LowLevelFEM.field_transformation(P2)

(sparse([2403, 3185, 3189, 3203, 3207, 3209, 3211, 3213, 3215, 5765  …  30848, 30852, 30839, 30841, 30843, 30845, 30840, 30842, 30844, 30846], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  30854, 30854, 30855, 30855, 30855, 30855, 30856, 30856, 30856, 30856], [1.0, -0.11718750000000006, 0.039062500000000014, 0.039062500000000014, -0.11718750000000006, 0.013732910156250014, -0.0045776367187500035, 0.001525878906250001, -0.0045776367187500035, 0.039062500000000014  …  -0.11840671663579501, 0.8840317166357948, 0.014020150544469454, -0.10467529296874999, 0.7815120760180302, -0.10467529296874999, 0.014020150544469454, -0.10467529296874999, 0.7815120760180302, -0.10467529296874999], 30856, 30856), sparse([14105, 14107, 14111, 14125, 14127, 14131, 14135, 14137, 14141, 14106  …  30855, 30840, 30842, 30844, 30846, 30848, 30850, 30852, 30854, 30856], [1, 1, 1, 1, 1, 1, 1, 1, 1, 2  …  30855, 30856, 30856, 30856, 30856, 30856, 30856, 30856, 30856, 30856], [-0.026914552635302895, -0.0034707760655282383, 0.0007

In [50]:
M = ∫(P2 ⋅ P2)
M[:,:]

30856×30856 SparseMatrixCSC{Float64, Int64} with 1088496 stored entries:
⎡⢿⣷⡗⣿⡏⠀⠱⡦⡟⠟⣿⠗⠓⠛⠙⢻⡾⠻⢓⢰⣧⠽⣷⢴⣴⣵⣮⣶⣷⣾⣾⣶⣶⣶⡶⣧⠁⠀⠀⠀⎤
⎢⣽⣭⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣯⎥
⎢⠋⠉⣿⣿⣿⣿⣝⣿⢟⢿⠻⡿⡯⡯⣛⣿⣿⣿⣽⣿⣫⢛⣹⣏⣫⠯⢿⢟⣏⠋⣍⠫⢝⢍⢩⣩⡯⣹⣙⣉⎥
⎢⠱⡦⣿⣿⣷⣽⣻⣾⣼⡯⢟⣽⣿⢭⢡⡶⡮⣾⠯⡸⣛⣫⣟⣟⢦⢼⣛⣗⡊⡩⠲⠒⠤⣃⠧⣐⢸⠤⡯⠏⎥
⎢⣿⠍⣿⣿⣿⣕⡶⡿⣿⣿⡟⣻⣹⣾⣛⡓⡾⣿⢋⣿⣴⢯⣈⣵⠦⡢⢟⡻⣧⣿⣀⠈⡅⠷⣈⡽⡼⣬⢙⡳⎥
⎢⢿⠟⣿⣿⣿⡦⣟⣵⣿⣩⡿⣯⣧⣿⢽⢾⣿⣚⡷⣅⣿⢦⣺⣯⡿⡿⣗⡕⣜⣘⡴⠉⡆⠆⠿⣞⢅⠋⢋⣏⎥
⎢⣽⠀⣿⣿⡯⡯⡟⣟⣳⣾⣭⣿⣿⣿⣯⣟⢿⡿⣾⢿⢏⢿⣸⣶⣥⠲⣙⣏⡪⣄⠫⠈⡃⡢⣴⣸⣝⣛⣊⠃⎥
⎢⣷⣀⣿⣿⣿⣼⢡⡶⢿⠸⣳⣗⣯⢿⢿⣷⣟⣿⣿⠿⡹⠿⡛⡽⠸⡱⣹⣫⣆⣚⡊⢘⠊⡪⣕⣆⣧⢐⠖⡄⎥
⎢⣾⡋⣿⣿⣿⣿⣪⣯⣾⣯⣻⢻⣿⡷⣿⣽⣻⣾⣦⣽⡼⢧⣭⣦⢗⣿⣮⣗⣏⡻⣪⢹⠟⢙⢽⢻⣹⣐⡿⣵⎥
⎢⢙⣐⣿⣿⣷⣿⣋⡣⣯⣴⠝⢯⣾⣟⣿⡟⣌⣿⡿⣯⣿⣽⣷⡿⢴⢿⣟⡼⣣⠦⣹⣟⣳⡸⢿⡆⣛⠶⠋⣯⎥
⎢⣍⡟⣿⣿⣯⢚⡿⣸⡴⣟⠻⣟⣯⣕⣷⡎⠶⣏⣟⣿⡿⣯⣷⣿⣷⡿⣟⣋⢙⣛⢉⠔⠦⠼⢧⣹⢟⠧⡾⡖⎥
⎢⢙⣟⣿⣿⡷⢾⣿⢽⢆⣼⡾⣾⢲⣾⣟⡬⠣⣿⣽⡿⣽⣿⣻⣾⡿⣯⣿⡯⢿⡼⡜⠽⢑⣛⣵⣫⣣⡥⣔⣂⎥
⎢⢔⣿⣿⣿⡯⡞⣈⣗⠨⡣⣿⡯⢡⡛⢖⡢⣽⣵⣴⣗⣽⡿⡿⣯⣿⣿⣻⣿⣽⣿⢸⡽⢷⣎⣇⣺⡯⣶⠇⠀⎥
⎢⢪⣿⣿⣿⣿⢗⢿⢼⣿⡱⢝⠽⡷⢼⡷⣺⢮⢿⣛⡽⡿⢹⡿⡿⣿⣾⣿⣿⣜⣷⣡⡎⢀⣙⣙⢻⣿⣻⡿⠻⎥
⎢⣹⣿⣿⣿⡯⠙⡎⡨⣭⣿⣒⢹⠊⢮⣨⢹⣯⡹⠩⡞⣷⢰⣛⡷⣷⣿⢶⣽⣿⣿⣿⣼⣝⣝⡿⡿⢭⠗⣯⠮⎥
⎢⢺⣿⣿⣿⡧⡙⢸⠂⡀⠘⡔⠋⡋⠂⣊⢈⣎⣚⣷⢾⢃⠔⣖⡍⣖⡶⡡⠾⣛⣿⢿⣷⣷⣺⢬⡇⡿⢓⣼⡇⎥
⎢⢸⣿⣿⣿⡗⢕⠤⢣⢥⡍⠨⠍⠩⡨⡪⡠⣟⢁⣙⡺⣈⡇⣵⢰⡹⢷⣄⢰⣗⢽⣹⣻⣿⣿⣿⣯⣉⣿⣿⣿⎥
⎢⠼⣯⣿⣿⡇⣲⢉⢣⣆⡼⣻⢧⣐⣻⠱⢽⣷⣓⠻⠷⣍⣳⡵⣻⣩⣹⣷⣘⣿⡯⠦⠷⡿⣿⣿⣿⣿⣿⡺⡷⎥
⎢⠁⠀⣿⣿⣏⣫⠒⡖⡒⣯⡥⠑⣷⢹⢉⢛⢓⢺⢻⡜⠿⡕⠍⡾⢫⣯⣿⣻⢧⠗⢿⢋⣧⣼⣿⣿⣿⣿⣿⣷⎥
⎣⠀⠀⡿⣿⡗⢸⡯⠏⢷⡰⡯⢴⠮⠘⠘⠥⢟⣯⡯⣤⢺⠯⠰⢹⠉⠁⣿⡋⡫⡟⠶⠿⣿⣿⢾⡮⢿⣿⣿⣿⎦

In [51]:
M = ∫(P2 ⋅ P2, gauss=:gll)
droptol!(M.A, 1e-10)
Ms = Ts' * M[:,:] * Ts
droptol!(Ms, 1e-13)
#dropzeros!(Ms)
(Ms)[1000:1008, 1000:1008]

9×9 SparseMatrixCSC{Float64, Int64} with 9 stored entries:
 7.75436   ⋅        ⋅        ⋅       …   ⋅        ⋅        ⋅        ⋅ 
  ⋅       2.62609   ⋅        ⋅           ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅       2.62609   ⋅           ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅       2.17086      ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅       …  5.99539   ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅       5.99539   ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅        ⋅       7.44837   ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅        ⋅        ⋅       7.44837

In [52]:
Ms

30856×30856 SparseMatrixCSC{Float64, Int64} with 30856 stored entries:
⎡⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⎦

In [53]:
@show nnz(Ts)
@show nnz(Rs)

@show Base.summarysize(K2.A) / 2^20
@show Base.summarysize(Ts) / 2^20
@show Base.summarysize(Rs) / 2^20

@show (Base.summarysize(Ts) + Base.summarysize(Rs)) /
      Base.summarysize(K2.A)

nnz(Ts) = 273240
nnz(Rs) = 273240
Base.summarysize(K2.A) / 2 ^ 20 = 33.453834533691406
Base.summarysize(Ts) / 2 ^ 20 = 4.404884338378906
Base.summarysize(Rs) / 2 ^ 20 = 4.404884338378906
(Base.summarysize(Ts) + Base.summarysize(Rs)) / Base.summarysize(K2.A) = 0.2633410728413056


0.2633410728413056

In [54]:
P2 = Field([mat];
    type=:VectorField,
    dim=2,
    fieldName=:u,
    basis=:spectral
)

T, R = spectralTransformation(P2; check=true)

@show T === P2.approximation.T
@show R === P2.approximation.R
@show norm(R*T - I, Inf)

M = ∫(P2 ⋅ P2; gauss=:gll)
Ms = T' * M.A * T
droptol!(Ms, 1e-14)

@show nnz(Ms)
@show size(Ms, 1)

T === P2.approximation.T = true
R === P2.approximation.R = true
norm(R * T - I, Inf) = 4.884981308350689e-15
nnz(Ms) = 32110
size(Ms, 1) = 30856


30856

In [55]:
saveField("test", u1)
u3 = loadField("test")

@show u3.model.basis
@show u3.model.reducedOrder
@show u3.model.approximation.T
@show u3.model.approximation.R
@show norm(u3.a - u1.a)

u3.model.basis = :lagrange
u3.model.reducedOrder = false
u3.model.approximation.T = nothing
u3.model.approximation.R = nothing
norm(u3.a - u1.a) = 0.0


0.0